In [17]:
# インポート
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import csv

In [18]:
# 設定
start_date_str = "20260501"
end_date_str = "20260504"
start_date = datetime.strptime(start_date_str, "%Y%m%d")
end_date = datetime.strptime(end_date_str, "%Y%m%d")
base_dir = Path("C:/keiba/tool/key/raw")
output_file = Path(f"C:/keiba/tool/key/integrated_key_data_{start_date_str}_{end_date_str}.csv") # 出力先ファイル名

In [19]:
# rawデータ読み込みとCSVファイル作成

all_data_list = []

current_date = start_date
while current_date <= end_date:
    year_dir = current_date.strftime("%Y")
    file_name = f"KEYnar{current_date.strftime('%Y%m%d')}.csv"
    file_path = base_dir / year_dir / file_name
    
    if file_path.exists():
        print(f"読み込み中: {file_path}")
        try:
            with open(file_path, mode='r', encoding='cp932', newline='') as f:
                reader = csv.reader(f)
                for row in reader:
                    if not row: continue
                    
                    # レースIDをnetkeibaと同じ形式に変換
                    race_id_raw = str(row[0])
                    race_id = (race_id_raw[0:4] + race_id_raw[8:10] + 
                               race_id_raw[4:6] + race_id_raw[6:8] + 
                               race_id_raw[14:16])
                    
                    # 2列目以降を2列ずつループ処理
                    horse_num = 1
                    for i in range(1, len(row), 2):
                        try:
                            key_score = row[i]
                            key_rank = row[i+1] if i+1 < len(row) else None
                            
                            # 指数が入っている場合のみリストに追加
                            if key_score and str(key_score).strip():
                                all_data_list.append({
                                    "race_id": race_id,
                                    "horse_number": horse_num,
                                    "key_index": key_score,
                                    "key_rank": key_rank
                                })
                        except IndexError:
                            break
                        horse_num += 1
            print(f"  完了：{file_name}")
        except Exception as e:
            print(f"  エラー発生 ({file_path}): {e}")
    else:
        print(f"スキップ: {file_path}")
    
    current_date += timedelta(days=1)

# --- CSV出力処理 ---
if all_data_list:
    final_df = pd.DataFrame(all_data_list)
    
    # 指数や順位を数値型に変換
    final_df["key_index"] = pd.to_numeric(final_df["key_index"], errors='coerce')
    final_df["key_rank"] = pd.to_numeric(final_df["key_rank"], errors='coerce')
    
    # CSVファイルとして書き出し
    final_df.to_csv(output_file, index=False, encoding="cp932")
    
    print("\n" + "="*30)
    print(f"処理が正常に終了しました。")
    print(f"保存先: {output_file}")
    print(f"合計レコード数: {len(final_df)} 件")
    print("="*30)
else:
    print("対象期間にデータが存在しませんでした。")

読み込み中: C:\keiba\tool\key\raw\2026\KEYnar20260501.csv
  完了：KEYnar20260501.csv
読み込み中: C:\keiba\tool\key\raw\2026\KEYnar20260502.csv
  完了：KEYnar20260502.csv
読み込み中: C:\keiba\tool\key\raw\2026\KEYnar20260503.csv
  完了：KEYnar20260503.csv
読み込み中: C:\keiba\tool\key\raw\2026\KEYnar20260504.csv
  完了：KEYnar20260504.csv

処理が正常に終了しました。
保存先: C:\keiba\tool\key\integrated_key_data_20260501_20260504.csv
合計レコード数: 1768 件
